# Business question: How can movie sales/success be predicted based on the language of reviews, the marketing aspects, and movie production elements?

# Suggested sub questions:
Language of Reviews: Can the sentiment and semantic themes (extracted via Transformer embeddings) from expert and user reviews accurately predict the box office tier of a movie?

Marketing Aspects: To what extent do the number of theaters and "opening weekend" performance correlate with long-term international box office success?

Production Elements: Which production features (Genre, Runtime, Rating, and Studio) are the strongest predictors of a movie’s financial success?

Model Performance: Which model (Baseline, Random Forest, or Neural Network) provides the highest accuracy in predicting movie success using these combined features?

In [ ]:
# Uploading the libraries that we need


import seaborn as sns # for visualization
import matplotlib.pyplot as plt # for visualization
from sklearn.impute import KNNImputer # to be used for KNN (runtime imputation)
import pandas as pd





In [ ]:
# Importing the datasets that we need to work with.

df_sales = pd.read_excel("../metacritic-dataset/sales.xlsx")
df_meta = pd.read_excel("../metacritic-dataset/metaClean43Brightspace.xlsx")
df_user = pd.read_excel("../metacritic-dataset/UserReviews.xlsx")
df_expert = pd.read_excel("../metacritic-dataset/ExpertReviews.xlsx")

expert_features = pd.read_csv("../transformer-data/processed/expert_features_with_svd.csv")
user_features   = pd.read_csv("../transformer-data/processed/user_features_with_svd.csv")


# Transformer
## Transformer-Based Feature Extraction (with Test Mode)

Transformer models (DistilBERT) are used to extract **sentiment** and **semantic embeddings**
from expert and user reviews. Sentiment analysis assigns polarity and confidence scores,
while embeddings capture deeper contextual meaning beyond simple word frequencies.

Running Transformer-based sentiment analysis and, in particular, generating embeddings
for all reviews is computationally expensive and can take **several hours** on the full
dataset.

⚠️ **Test Mode Notice**
To allow faster execution and demonstrate that the pipeline works end-to-end, a
**TEST MODE** is included. When enabled, only a small subset of reviews (e.g., 500 records)
is processed and outputs are saved to a separate directory.

The full pipeline has already
been executed on the complete dataset, and the final **Sentiment**, **Embedding**, and
**SVD-compressed** feature files are provided with this project.



In [ ]:
# ==================================================
# TEST MODE CONFIGURATION
# ==================================================
TEST_MODE = True        # ← set to False for full run
N_TEST_ROWS = 500       # number of reviews to process in test mode

if TEST_MODE:
    PROCESSED_DIR = "../transformer-data/processed_test"
    print("⚠️ TEST MODE ENABLED")
    print(f"Using only {N_TEST_ROWS} rows")
else:
    PROCESSED_DIR = "../transformer-data/processed"


SUFFIX = "_test" if TEST_MODE else ""


## Step 1: Import Required Libraries

This step loads all libraries required for sentiment analysis, transformer
embeddings, aggregation, and dimensionality reduction.


In [ ]:
# Core libraries
import os
import gc
import pandas as pd
import numpy as np

# NLP / Deep Learning
import torch
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModel

# Dimensionality Reduction
from sklearn.decomposition import TruncatedSVD


## Step 2: Load Raw Review Datasets

These are the original expert and user review files.
No processing is applied at this stage.


In [ ]:
# Load raw review data (test mode: 500 rows)
df_expert_transformer = df_expert
df_user_transformer   = df_user

# Apply test mode row limit BEFORE any NLP work
if TEST_MODE:
    df_expert_transformer = df_expert.head(N_TEST_ROWS)
    df_user_transformer   = df_user.head(N_TEST_ROWS)



# Directory where processed outputs will be saved
os.makedirs(PROCESSED_DIR, exist_ok=True)


## Step 3: Text Cleaning Function

This function matches the original script exactly.
It ensures all review text is safe for Transformer models.


In [ ]:
def clean_text(series):
    return (
        series.astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip(" '\"")
    )


## Step 4: Sentiment Analysis Function

Uses DistilBERT fine-tuned on SST-2.
Logic and parameters match the original script.


In [ ]:
def run_sentiment(df, text_col="Rev"):
    device = 0 if torch.cuda.is_available() else -1
    print("Using GPU" if device == 0 else "Using CPU")

    sentiment = pipeline(
        "sentiment-analysis",
        model="distilbert-base-uncased-finetuned-sst-2-english",
        device=device,
        truncation=True,
        max_length=512,
        torch_dtype=torch.float16 if device == 0 else torch.float32
    )

    batch_size = 128 if device == 0 else 64
    outputs = []

    for i in tqdm(range(0, len(df), batch_size), desc="Sentiment"):
        outputs.extend(sentiment(df[text_col].iloc[i:i + batch_size].tolist()))

    df["sentiment_label"] = [o["label"] for o in outputs]
    df["sentiment_score"] = [
        o["score"] if o["label"] == "POSITIVE" else -o["score"]
        for o in outputs
    ]

    del sentiment, outputs
    gc.collect()
    torch.cuda.empty_cache()

    return df


## Step 5: Run Sentiment Analysis for Expert Reviews


In [ ]:
expert_sentiment_path = f"{PROCESSED_DIR}/expert_sentiment{SUFFIX}.csv"

if not os.path.exists(expert_sentiment_path):

    df = df_expert_transformer.copy()
    df["Rev"] = clean_text(df["Rev"])
    print(f"Expert reviews: {len(df)}")

    df = run_sentiment(df)
    df.to_csv(expert_sentiment_path, index=False)

    print("✔ Expert sentiment saved")

else:
    print("✔ Expert sentiment already exists — skipping")


## Step 6: Run Sentiment Analysis for User Reviews


In [ ]:
user_sentiment_path = f"{PROCESSED_DIR}/user_sentiment{SUFFIX}.csv"

if not os.path.exists(user_sentiment_path):

    df = df_user_transformer.copy()
    df["Rev"] = clean_text(df["Rev"])
    print(f"User reviews: {len(df)}")

    df = run_sentiment(df)
    df.to_csv(user_sentiment_path, index=False)

    print("✔ User sentiment saved")

else:
    print("✔ User sentiment already exists — skipping")


## Step 7: Transformer Embedding Function

Uses DistilBERT and mean pooling.
Logic is identical to the original script.


In [ ]:
def run_embeddings(df, prefix):
    device = 0 if torch.cuda.is_available() else -1

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModel.from_pretrained(
        "distilbert-base-uncased"
    ).to("cuda" if device == 0 else "cpu")
    model.eval()

    emb_batch = 64 if device == 0 else 32
    all_embeddings = []

    for i in tqdm(range(0, len(df), emb_batch), desc=f"{prefix} embeddings"):
        batch = df["Rev"].iloc[i:i + emb_batch].tolist()

        with torch.no_grad():
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)

            emb = model(**encoded).last_hidden_state.mean(dim=1)
            all_embeddings.append(emb.cpu())

        torch.cuda.empty_cache()

    embeddings = torch.cat(all_embeddings).numpy()
    emb_dim = embeddings.shape[1]

    emb_cols = [f"{prefix}_emb_{i}" for i in range(emb_dim)]
    df = pd.concat([df.reset_index(drop=True),
                    pd.DataFrame(embeddings, columns=emb_cols)],
                   axis=1)

    del model, tokenizer, all_embeddings
    gc.collect()
    torch.cuda.empty_cache()

    return df, emb_cols


## Step 8: Generate Embeddings for Expert Reviews


In [ ]:
expert_embeddings_path = f"{PROCESSED_DIR}/expert_embeddings_raw{SUFFIX}.csv"

if not os.path.exists(expert_embeddings_path):

    df = pd.read_csv(expert_sentiment_path)
    df["Rev"] = clean_text(df["Rev"])  # critical safety step

    df, expert_emb_cols = run_embeddings(df, prefix="expert")
    df.to_csv(expert_embeddings_path, index=False)

    print("✔ Expert embeddings generated")

else:
    print("✔ Expert embeddings already exist — skipping")


## Step 9: Generate Embeddings for User Reviews


In [ ]:
user_embeddings_path = f"{PROCESSED_DIR}/user_embeddings_raw{SUFFIX}.csv"

if not os.path.exists(user_embeddings_path):

    df = pd.read_csv(user_sentiment_path)
    df["Rev"] = clean_text(df["Rev"])

    df, user_emb_cols = run_embeddings(df, prefix="user")
    df.to_csv(user_embeddings_path, index=False)

    print("✔ User embeddings generated")

else:
    print("✔ User embeddings already exist — skipping")


## Step 10: Aggregation Function

Aggregates review-level features to the movie (URL) level.
This logic is identical to the original script.


In [ ]:
def aggregate_features(df, emb_cols, prefix):
    return (
        df.groupby("url")
        .agg(
            **{
                f"{prefix}_sentiment_mean": ("sentiment_score", "mean"),
                f"{prefix}_positive_ratio": ("sentiment_label",
                                             lambda x: (x == "POSITIVE").mean()),
                f"{prefix}_review_count": ("sentiment_score", "count"),
                **{c: (c, "mean") for c in emb_cols}
            }
        )
        .reset_index()
    )


## Step 11: Aggregate Expert Features


In [ ]:
expert_features_path = f"{PROCESSED_DIR}/expert_features_with_embeddings{SUFFIX}.csv"

if not os.path.exists(expert_features_path):

    df = pd.read_csv(expert_embeddings_path)
    emb_cols = [c for c in df.columns if c.startswith("expert_emb_")]

    expert_features = aggregate_features(df, emb_cols, "expert")
    expert_features.to_csv(expert_features_path, index=False)

    print("✔ Expert features aggregated")

else:
    print("✔ Expert aggregated features already exist — skipping")


## Step 12: Aggregate User Features


In [ ]:
user_features_path = f"{PROCESSED_DIR}/user_features_with_embeddings{SUFFIX}.csv"

if not os.path.exists(user_features_path):

    df = pd.read_csv(user_embeddings_path)
    emb_cols = [c for c in df.columns if c.startswith("user_emb_")]

    user_features = aggregate_features(df, emb_cols, "user")
    user_features.to_csv(user_features_path, index=False)

    print("✔ User features aggregated")

else:
    print("✔ User aggregated features already exist — skipping")


## Step 13: Truncated SVD Function

Reduces embedding dimensionality while preserving variance.


In [ ]:
def run_svd(df, emb_prefix, svd_prefix, n_components=50):

    emb_cols = [c for c in df.columns if c.startswith(emb_prefix)]
    meta_cols = [c for c in df.columns if c not in emb_cols]

    print(f"Embedding columns found: {len(emb_cols)}")

    X = df[emb_cols].values
    X_scaled = StandardScaler().fit_transform(X)

    # ✅ CRITICAL FIX: cap components for small datasets
    max_components = min(n_components, X_scaled.shape[0], X_scaled.shape[1])

    svd = TruncatedSVD(
        n_components=max_components,
        random_state=42
    )

    X_svd = svd.fit_transform(X_scaled)

    explained = svd.explained_variance_ratio_.sum()
    print(f"Explained variance retained: {explained:.2%}")
    print(f"SVD components used: {max_components}")

    svd_cols = [f"{svd_prefix}{i}" for i in range(max_components)]
    df_svd = pd.DataFrame(X_svd, columns=svd_cols)

    df_final = pd.concat(
        [
            df[meta_cols].reset_index(drop=True),
            df_svd.reset_index(drop=True)
        ],
        axis=1
    )

    return df_final


## Step 14: Apply SVD and Save Final Outputs


In [ ]:
expert_svd_path = f"{PROCESSED_DIR}/expert_features_with_svd{SUFFIX}.csv"

if not os.path.exists(expert_svd_path):
    df = pd.read_csv(expert_features_path)
    df = run_svd(df, "expert_emb_", "expert_svd_")
    df.to_csv(expert_svd_path, index=False)
    print("✔ Expert SVD saved")
else:
    print("✔ Expert SVD already exists — skipping")


In [ ]:
user_svd_path = f"{PROCESSED_DIR}/user_features_with_svd{SUFFIX}.csv"

if not os.path.exists(user_svd_path):
    df = pd.read_csv(user_features_path)
    df = run_svd(df, "user_emb_", "user_svd_")
    df.to_csv(user_svd_path, index=False)
    print("✔ User SVD saved")
else:
    print("✔ User SVD already exists — skipping")


# Transformer is done

In [ ]:
# Check our data frames and data types.

df_sales.info()
df_meta.info()
df_user.info()
df_expert.info()

expert_features.info()
user_features.info()

# df_sales.tale(10) # just to check 
# df_sales.head(10)


In [ ]:
df_sales.head(3)

In [ ]:
df_sales.shape

In [ ]:
df_sales.dtypes

In [ ]:
df_sales.describe()

In [ ]:
df_sales.isna().sum()

In [ ]:
df_sales.duplicated().sum()

In [ ]:
df_sales.columns

Cleaning data:

In [ ]:
# Function to clean movie titles in Sales and Meta tables (since sales table doesn't use the same URLs and both dataframes have title columns in them). Code by Muayyad 
def clean_title(text):
    if pd.isna(text):
        return ""
    # Convert to string, lowercase, and remove extra spaces
    return str(text).lower().strip()

# Apply cleaning to both dataframes
df_sales['title_clean'] = df_sales['title'].apply(clean_title)
df_meta['title_clean'] = df_meta['title'].apply(clean_title)

In [ ]:
# cleaning meta and sales tables and datatypes. Code by Muayyad 

# function to fix garbled names (used before in DB course)
def fix_names(name):
    if not isinstance(name, str) or name.lower() == 'nan':
        return "Unknown"
    
    # Dictionary of common garbled characters (used before in DB course)
    replacements = {
        'Ã©': 'é', 'Ãª': 'ê', 'Ã¨': 'è', 'Ã§': 'ç', 'Ã ': 'à', 'Ã¡': 'á', 
        'Ã±': 'ñ', 'Ã¶': 'ö', 'Ã¼': 'ü', 'ã³': 'ó', 'Ã°': 'ð', 'Ãº': 'ú'
    }
    
    for pattern, replacement in replacements.items():
        name = name.replace(pattern, replacement)
    
    # Clean extra spaces and make it "Title Case"
    return " ".join(name.split()).title()



# Cleaning df_sales and the numerical columns:

# Drop entirely empty rows and columns
df_sales.dropna(how='all', axis=0, inplace=True) 
df_sales.dropna(how='all', axis=1, inplace=True) 

# Cleaning and fixing datatypes
sales_money_cols = [
    'international_box_office', 'domestic_box_office', 'worldwide_box_office', 
    'production_budget', 'opening_weekend'
]
for col in sales_money_cols:
    if col in df_sales.columns:
        df_sales[col] = df_sales[col].astype(str).str.replace(r'[$,]', '', regex=True)
        df_sales[col] = pd.to_numeric(df_sales[col], errors='coerce').fillna(0)

# Fix other numerical columns
sales_numeric_cols = ['year', 'theatre_count', 'avg run per theatre', 'runtime']
for col in sales_numeric_cols:
    if col in df_sales.columns:
        df_sales[col] = pd.to_numeric(df_sales[col], errors='coerce').fillna(0)

# Keep only the month in release_date (we might use this to create a season feature)
df_sales['release_date'] = df_sales['release_date'].str.split().str[0]

# Standardize the titles
df_sales['title'] = df_sales['title'].astype(str).str.strip()

# cleaning df_meta:

# Drop entirely empty rows and columns
df_meta.dropna(how='all', axis=0, inplace=True)
df_meta.dropna(how='all', axis=1, inplace=True)

# Clean rating column and remove '|'
df_meta['rating'] = df_meta['rating'].astype(str).str.replace('|', '', regex=False).str.strip()
df_meta['rating'] = df_meta['rating'].replace(['nan', ''], 'Unrated')

# Fix meta score and user score
df_meta['metascore'] = pd.to_numeric(df_meta['metascore'], errors='coerce').fillna(0)
df_meta['userscore'] = pd.to_numeric(df_meta['userscore'], errors='coerce').fillna(0)

# Clean runtime column
df_meta['runtime'] = df_meta['runtime'].astype(str).str.extract('(\d+)').astype(float).fillna(0)

# Fix (Director and Cast) names using the fix names function
df_meta['director'] = df_meta['director'].apply(fix_names)
df_meta['cast'] = df_meta['cast'].apply(fix_names)

# Fix release date and extract year (we'll use this to join movies based on title + year)
df_meta['RelDate'] = pd.to_datetime(df_meta['RelDate'], errors='coerce')
df_meta['year'] = df_meta['RelDate'].dt.year

# Fill missing summary text
df_meta['summary'] = df_meta['summary'].fillna("No summary available")

print("Both DataFrames cleaned according to their specific columns!")


In [ ]:
df_sales.columns

Initial EDA for all sales data before the merge:

In [ ]:
# Set the style for a professional look. Code by Muayyad 
sns.set_theme(style="whitegrid")

# Create a figure with 3 subplots in 1 row
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Distribution of Worldwide Box Office (Log Scale)
# This shows us the scale of the movies—most are small, few are blockbusters.
sns.histplot(df_sales[df_sales['worldwide_box_office'] > 0]['worldwide_box_office'], 
             bins=30, kde=True, ax=axes[0], log_scale=True, color='teal')
axes[0].set_title('Distribution of Box Office (Log Scale)')
axes[0].set_xlabel('Worldwide Revenue ($)')

# 2. Box Office Trends Over Time
# Shows how the industry has grown from the 30k records.
yearly_revenue = df_sales.groupby('year')['worldwide_box_office'].median()
axes[1].plot(yearly_revenue.index, yearly_revenue.values, marker='o', color='darkorange')
axes[1].set_title('Median Box Office per Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Median Revenue ($) (Worldwide box office)')

# 3. Production Budget vs. Worldwide Box Office
# A scatter plot to see the correlation between spending and earning.
sns.scatterplot(data=df_sales[df_sales['production_budget'] > 0], 
                x='production_budget', y='worldwide_box_office', 
                alpha=0.2, ax=axes[2], color='purple')
axes[2].set_title('Budget vs. Worldwide Revenue')
axes[2].set_xscale('log')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

1. The first graph shows that the market is dominated by a few successful movies (Blockbusters).
2. In the second graph we used the median to eleminate the effect of outliers, the decline around the year 2020 is consistent with the global impact of covid pandemic. (we'll investigate or drop the 0 values going further)
3. The last chart shows that there is a correlation between budget and world wide box office, but there is also a big variation, meaning that we need to look at other factors like "reviews" and "genres" to predict a blockbuster with our AI models.


### Outlier Analysis ---- Ahmed
To better understand extreme values in key numerical variables, boxplots are used to identify potential outliers in movie runtime, production budget, and box office performance. These variables are known to be right-skewed in real-world movie datasets and may require transformation to ensure better modeling.


In [ ]:
#  Visualize outliers using boxplots ------ Ahmed
#numeric_cols = ['runtime', 'production_budget', 'worldwide_box_office']
numeric_cols = ['runtime', 'production_budget', 'worldwide_box_office'] # specify the numerical columns to analyze

plt.figure(figsize=(14, 5))

for i, col in enumerate(numeric_cols, 1):
    plt.subplot(1, 3, i)
    sns.boxplot(y=df_sales[col])
    plt.title(f'Boxplot of {col}')

plt.tight_layout()
plt.show()
# Observations:
# Runtime:few movies have extremely long runtimes, which could skew analyses
# production_budget:several movies with very high budgets, indicating potential outliers
# worldwide_box_office:few movies with exceptionally high box office earnings
# -----These outliers need to be addressed before modeling-----

The boxplots:
reveal extreme values, particularly for budget and box office revenue. These represent blockbuster movies rather than data errors. Removing them would eliminate meaningful observations, therefore a logarithmic transformation is applied to reduce skewness while preserving relative differences.
Ahmed


In [ ]:
# Log transformation applied to reduce skewness identified during EDA ------ Ahmed

df_sales['log_production_budget'] = np.log1p(df_sales['production_budget'])
df_sales['log_worldwide_box_office'] = np.log1p(df_sales['worldwide_box_office'])

# Visualize the transformed distributions
# Visualizing the effect of log transformation on budget
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(df_sales['production_budget'], bins=50)
plt.title('Original Budget Distribution')

plt.subplot(1, 2, 2)
sns.histplot(df_sales['log_production_budget'], bins=50)
plt.title('Log-Transformed Budget Distribution')

plt.tight_layout()
plt.show()
# Visualizing the effect of log transformation on box office
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(df_sales['worldwide_box_office'], bins=50)
plt.title('Original Box Office Distribution')

plt.subplot(1, 2, 2)
sns.histplot(df_sales['log_worldwide_box_office'], bins=50)
plt.title('Log-Transformed Box Office Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Creating a figure with 2 subplots side-by-side for the two features that we see important. Code by Muayyad 
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Top 10 Genres by total revenue
top_genres = df_sales.groupby('genre')['worldwide_box_office'].sum().sort_values(ascending=False).head(10)
sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[0], palette='viridis')
axes[0].set_title('Top 10 Genres by Total Worldwide Revenue')
axes[0].set_xlabel('Total Revenue (Billions)')

# Average revenue by creative type
creative_type = df_sales.groupby('creative_type')['worldwide_box_office'].mean().sort_values(ascending=False)
sns.barplot(x=creative_type.values, y=creative_type.index, ax=axes[1], palette='magma')
axes[1].set_title('Average Revenue by Creative Type')
axes[1].set_xlabel('Average Revenue ($)')

plt.tight_layout()
plt.show()

The categorical analysis reveals that Adventure and Action genres lead in total market volume, while the Super Hero creative type is the strongest individual indicator of high average revenue. These two key features will be very useful to detect blockbusters in the AI model.


Creating a master table that has all the required features to be used for AI modeling in a way that answers the business question:

In [ ]:
# Data cleaning, and reviews aggregation, this will be used to add new features when making a master table. Code by Muayyad 

# Changing data types to numbers for some numerical columns in user and expert reviews dataframes:
df_user['idvscore'] = pd.to_numeric(df_user['idvscore'], errors='coerce')
df_expert['idvscore'] = pd.to_numeric(df_expert['idvscore'], errors='coerce')
df_user['thumbsUp'] = pd.to_numeric(df_user['thumbsUp'], errors='coerce')
df_user['thumbsTot'] = pd.to_numeric(df_user['thumbsTot'], errors='coerce')

# Aggregate one row per URL for user reviews:
user_agg = df_user.groupby('url').agg({'idvscore': 'mean', 'Rev': 'count','thumbsUp': 'sum','thumbsTot': 'sum'}).reset_index()
user_agg.columns = ['url', 'user_idvscore_avg', 'user_rev_count','user_thumbsUp_tot', 'user_thumbsTot_tot']

# Create the Thumbs Down column using formula: thumbstot-thumbsup and dropping thumbs total (we keep only negative and positive)
user_agg['user_thumbsDown_tot'] = user_agg['user_thumbsTot_tot'] - user_agg['user_thumbsUp_tot']
user_agg = user_agg.drop(columns=['user_thumbsTot_tot'])

# Aggregate one row per URL for expert reviews:
expert_agg = df_expert.groupby('url').agg({'idvscore': 'mean', 'Rev': 'count'}).reset_index()
expert_agg.columns = ['url', 'expert_score_avg', 'expert_rev_count']


In [ ]:
# Creating the master table. Code by Muayyad 

# Standardize titles for the merge
df_sales['title_clean'] = df_sales['title'].str.lower().str.strip()
df_meta['title_clean'] = df_meta['title'].str.lower().str.strip()

# Merge Meta and Sales
df_master = pd.merge(
    df_meta, 
    df_sales, 
    on=['title_clean', 'year'], 
    how='inner', 
    suffixes=('', '_sales')
)

# Join the aggregated reviews tables
df_master = pd.merge(df_master, user_agg, on='url', how='left')
df_master = pd.merge(df_master, expert_agg, on='url', how='left')

# --------------------------------------------------
# MERGE EXPERT + USER EMBEDDINGS (ON URL)
# --------------------------------------------------
features_merged = pd.merge(
    expert_features,
    user_features,
    on='url',
    how='inner'
)

df_master = pd.merge(
    df_master,
    features_merged,
    on='url',
    how='left'
)


# Create unique Movie ID
df_master['movie_id'] = range(1, len(df_master) + 1)

# Handle missing review data by filling NaNs with 0 (this is useful for the AI model)
review_cols = [
    'user_idvscore_avg', 'user_rev_count', 'user_thumbsUp_tot', 
    'user_thumbsDown_tot', 'expert_score_avg', 'expert_rev_count'
]
df_master[review_cols] = df_master[review_cols].fillna(0)

# Final cleanup of duplicated columns
if 'title_sales' in df_master.columns:
    df_master.drop(columns=['title_sales'], inplace=True)

print(f"New Master Table created with {len(df_master)} movies.")
df_master[['title', 'year', 'user_thumbsUp_tot', 'user_thumbsDown_tot', 'worldwide_box_office']].head()

In [ ]:
# I want to add the movie ID back to the original dataframes as foreign key for better data integrity. Code by Muayyad 
# Create mapping with year and title
id_mapping = df_master[['title_clean', 'year', 'movie_id']].drop_duplicates()

# Add movie_id to df_sales
df_sales = pd.merge(df_sales, id_mapping, on=['title_clean', 'year'], how='left')

# Add movie_id to df_meta
df_meta = pd.merge(df_meta, id_mapping, on=['title_clean', 'year'], how='left')

# Add movie_id to the reviews datasets (using the 'url' which is unique)
# Create mapping with URL
url_mapping = df_master[['url', 'movie_id']].drop_duplicates()

df_user = pd.merge(df_user, url_mapping, on='url', how='left')
df_expert = pd.merge(df_expert, url_mapping, on='url', how='left')

df_sales.head(3) # check


EDA and feature engineering for the master table:

In [ ]:
# check data types and missing data. Code by Muayyad 
pd.concat([df_master.dtypes, df_master.isna().sum()], axis=1, keys=['Data Type', 'Missing data count'])

In [ ]:
# Convert text columns we need to categorical.. Code by Muayyad 
df_master['rating'] = df_master['rating'].astype('category')
df_master['release_date'] = df_master['release_date'].astype('category') # because previously we only kept the year in this column
df_master['genre'] = df_master['genre'].astype('category')
df_master['studio'] = df_master['studio'].astype('category')
df_master['creative_type'] = df_master['creative_type'].astype('category')

# Convert floats to integers for columns that are "counts"
# better for the financial columns too since we have no decimal places
cols_to_int = ['production_budget', 'worldwide_box_office', 'theatre_count', 
               'user_thumbsUp_tot', 'user_thumbsDown_tot', 'metascore']

df_master[cols_to_int] = df_master[cols_to_int].astype('int64')

df_master.dtypes # check

In [ ]:
# Set up a 2x2 grid for a nicer visualization. Code by Muayyad 
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Top-Left: Correlation Heatmap
cols_to_corr = [
    'worldwide_box_office', 'production_budget', 'metascore', 
    'user_idvscore_avg', 'user_thumbsUp_tot', 'user_thumbsDown_tot', 
    'expert_score_avg', 'runtime'
]
sns.heatmap(df_master[cols_to_corr].corr(), annot=True, cmap='coolwarm', fmt='.2f', ax=axes[0,0])
axes[0,0].set_title('Correlation Heatmap: Drivers of Success')

# Top-Right: Budget vs Revenue (Strongest Correlation: 0.79)
sns.regplot(data=df_master, x='production_budget', y='worldwide_box_office', 
            scatter_kws={'alpha':0.2, 'color':'blue'}, line_kws={'color':'black'}, ax=axes[0,1])
axes[0,1].set_yscale('log')
axes[0,1].set_xscale('log')
axes[0,1].set_title('Production Budget vs. Worldwide Revenue (Log-Log Scale)')

# Bottom-Left: Thumbs Up vs Revenue
sns.regplot(data=df_master, x='user_thumbsUp_tot', y='worldwide_box_office', 
            scatter_kws={'alpha':0.2, 'color':'green'}, line_kws={'color':'black'}, ax=axes[1,0])
axes[1,0].set_yscale('log')
axes[1,0].set_title('User Thumbs Up vs. Worldwide Revenue (Log Scale)')

# Bottom-Right: Thumbs Down vs Revenue
sns.regplot(data=df_master, x='user_thumbsDown_tot', y='worldwide_box_office', 
            scatter_kws={'alpha':0.2, 'color':'red'}, line_kws={'color':'black'}, ax=axes[1,1])
axes[1,1].set_yscale('log')
axes[1,1].set_title('User Thumbs Down vs. Worldwide Revenue (Log Scale)')

plt.tight_layout()
plt.savefig('eda_master_grid.png')

The correlation heatmap shows that spending money on the budget is the most reliable way to predict how much money a movie will make.
Production budget has the most correlation with the world-wide-revenue.
Thumbs-up and Thumbs-down both represent engagement/audience reach, that means most successful movies get alot of hate, or love!

Since we won't use all the features for our AI models, we'll create a final dataset and we'll keep only the features that we see useful, we'll also keep only Worldwide box office as our "Y" where the value is more than "0" to train our AI model, we'll also discard international and domestic box office as we won't use those to prevent data leakage.

We'll keep the records of the unknown world wide box office for later so we can use our developed model to predict it, we'll remove them from the training data when we finish feature engineering.

In [ ]:
# Define the columns we want to keep. Code by Muayyad

expert_svd_cols = [c for c in df_master.columns if c.startswith("expert_svd_")]
user_svd_cols   = [c for c in df_master.columns if c.startswith("user_svd_")]


features_to_keep = [
    'worldwide_box_office', 'production_budget', 'theatre_count',
    'runtime', 'metascore', 'userscore', 'expert_score_avg',
    'user_thumbsUp_tot', 'user_thumbsDown_tot', 'user_rev_count',
    'rating', 'genre', 'release_date', 'year', 'creative_type', 'movie_id', 'director',
    'expert_review_count', 'expert_positive_ratio', 'expert_sentiment_mean',
    'user_review_count', 'user_positive_ratio', 'user_sentiment_mean'
] + expert_svd_cols + user_svd_cols

# Create the final modeling dataframe
df_final = df_master[features_to_keep].copy()

df_final.tail(5)

In [ ]:
# check file types and missing data for our final table. Code by Muayyad 
pd.concat([df_final.dtypes, df_final.isna().sum()], axis=1, keys=['Data Type', 'Missing data count'])

# check length of the dataframe (how many records)
# print('original length of dataframe:', len(df_final)) # code inspired by Linear-Logistic-Regression code

Worldwide_box_office will be our target (Y) and the others are our features (X).
We'll make some more feature engineering below:

In [ ]:
# checking how many times each genre appears in the dataset, split the coma seperated genres (extracted initially from meta table). Code by Muayyad 
df1 = df_final['genre_list'] = df_final['genre'].str.split(',')
df1 = df_final.explode('genre_list') # to put seperated genres in columns

In [ ]:
# Group genres to see the most frequent ones
df1.groupby('genre_list')['genre_list'].count().sort_values(ascending=False)

In [ ]:
# We'll select the first 10 genres, because from our initial EDA we know that Adventure is the most successful genre, so it is important for the prediction. Code by Muayyad 
# Prepare the list and the Top 10 Target Genres
df_final['genre_list'] = df_final['genre'].astype(str).str.split(',')
main_genres = [
    'Drama', 'Comedy', 'Thriller', 'Action', 'Romance', 
    'Documentary', 'Adventure', 'Horror', 'Crime', 'Sci-Fi'
]

# this piece of code was enhanced with AI instead of the very long logic we used
for g in main_genres:
    # Clean the genre name for the column header (remove spaces/dashes)
    col_name = f'is_{g.lower().replace(" ", "_").replace("-", "_")}'
    
    # Check if the genre exists in the list for each row
    # .apply handles the list check; .astype(int) turns True/False into 1/0
    df_final[col_name] = df_final['genre_list'].apply(lambda x: g in [i.strip() for i in x]).astype('int64')

# Add "is_other_genre" (count of genres NOT in our top 10)
df_final['is_other_genre'] = df_final['genre_list'].apply(
    lambda x: len([i.strip() for i in x if i.strip() not in main_genres])
).astype('int64')

# Add "nr_genres" (total count of genres for that movie)
df_final['nr_genres'] = df_final['genre_list'].apply(lambda x: len(x)).astype('int64')

# Drop the temporary list column and the genre (the source column after encoding) to keep df_final clean
df_final = df_final.drop(columns=['genre_list','genre'])

print("Top 10 Genre features created successfully!")
print(df_final.filter(like='is_').head())

In [ ]:
# Create season map to create a new feature for the season.. Code by Muayyad 
season_map = {
    'January': 'Winter', 'February': 'Winter', 'December': 'Winter',
    'March': 'Spring', 'April': 'Spring', 'May': 'Spring',
    'June': 'Summer', 'July': 'Summer', 'August': 'Summer',
    'September': 'Autumn', 'October': 'Autumn', 'November': 'Autumn'
}

# Map the seasons into df_final 
# (We use .str.strip() just in case there are hidden spaces in the month names)
df_final['season'] = df_final['release_date'].str.strip().map(season_map)

# One-Hot Encode the seasons (Convert to is_summer, is_winter, etc.)
season_dummies = pd.get_dummies(df_final['season'], prefix='is_season').astype('int64')

# Join the new columns and drop the original text 'season' and 'release_date'
df_final = pd.concat([df_final, season_dummies], axis=1)
df_final = df_final.drop(columns=['season', 'release_date'])

print(df_final.filter(like='is_season').head())

In [ ]:
df_final.head(5)

In [ ]:
df_final.info()

In [ ]:
# check Nan values vs "0" in Runtime to impute
nan_count = df_final['runtime'].isna().sum()
zero_count = (df_final['runtime'] == 0).sum()

print(f"Number of Nan values: {nan_count}")
print(f"Number of 0.0 runtimes: {zero_count}")

In [ ]:


# Flag the rows that are missing runtime so we can look at them later. Code by Muayyad 
missing_mask = (df_final['runtime'] == 0) | (df_final['runtime'].isna())

# Convert the 0.0 values to NaN for the imputer
df_final['runtime'] = df_final['runtime'].replace(0, np.nan)

# Select columns to take into consideration # code enhanced with AI instead of the manual naming of the previously encoded columns
impute_cols = ['runtime', 'production_budget'] + [col for col in df_final.columns if 'is_' in col]

# Initialize and run the Imputer, we'll check the nearest 3 neighbours
imputer = KNNImputer(n_neighbors=3)
df_temp = pd.DataFrame(imputer.fit_transform(df_final[impute_cols]), columns=impute_cols)

# We create a small table showing the "Before" (which was 0/NaN) and the "After"
verification_df = pd.DataFrame({
    'Old_Runtime': 0, # this was changed to 0 earlier when we were changing data types
    'Imputed_Runtime': df_temp.loc[missing_mask, 'runtime'].values
})

print("Check the imputation")
print(verification_df.head(10)) 
# This lets you see if the numbers look realistic (for example, 90, 105, 120)




In [ ]:
# The numbers actually look realistic. Code by Muayyad 
# Put the verified runtimes back into df_final
df_final['runtime'] = df_temp['runtime'].values

# Final cleanup: round to whole minutes
df_final['runtime'] = df_final['runtime'].round(0).astype('int64')

del df_temp

print("Final check number of zeros =", (df_final['runtime'] == 0).sum())

# Preparing for modelling

we'll create a new table called df_final_clean that has values greater than 0 for world wide box office, and move the ones that has 0 values to another dataframe to predict them later

In [ ]:
df_ready = df_final.copy()

In [ ]:
# --------------------------------------------------
# STEP 2: Target variable
# --------------------------------------------------

y = df_ready['worldwide_box_office']

print("Target min:", y.min())
print("Target max:", y.max())
print("Target missing values:", y.isna().sum())

In [ ]:
# ==============================
# CLEAN RATING
# ==============================

rating_map = {
    'Not Rated': 'Unrated',
    'NR': 'Unrated',
    'TV-MA': 'R',
    'TV-14': 'PG-13',
    'TV-G': 'G',
    'TV-PG': 'PG'
}

df_ready['rating'] = (
    df_ready['rating']
    .replace(rating_map)
    .fillna('Unrated')
)

print("Rating distribution after cleaning:")
print(df_ready['rating'].value_counts())
print("-" * 50)

In [ ]:
# ==============================
# IMPUTE CREATIVE TYPE
# ==============================

# Safe categorical imputation (MODE)
df_ready['creative_type'] = (
    df_ready['creative_type']
    .fillna(df_ready['creative_type'].mode()[0])
)

print("Creative type missing values after imputation:",
      df_ready['creative_type'].isna().sum())
print("-" * 50)


In [ ]:
# ==============================
# FIX DIRECTOR
# Convert director identity → experience signal
# ==============================

# Count how many movies each director has
director_counts = df_ready['director'].value_counts()

# Map experience buckets
df_ready['director_bucket'] = df_ready['director'].map(
    lambda x:
        'new_director' if director_counts.get(x, 0) == 1 else
        'low_exp' if director_counts.get(x, 0) <= 3 else
        'mid_exp' if director_counts.get(x, 0) <= 10 else
        'high_exp'
)

# Drop high-cardinality original column
df_ready = df_ready.drop(columns=['director'])

print("Director experience buckets:")
print(df_ready['director_bucket'].value_counts())
print("-" * 50)




In [ ]:
# ==============================
# ONE-HOT ENCODING
# ==============================

df_ready = pd.get_dummies(
    df_ready,
    columns=['rating', 'creative_type', 'director_bucket'],
    drop_first=True
)

print("Encoding complete.")
print("Final shape:", df_ready.shape)
print("-" * 50)

# ==============================
# FINAL SANITY CHECK
# ==============================

print("Dtype summary:")
print(df_ready.dtypes.value_counts())


print("STEP 3 COMPLETED — DATA IS CLEAN & MODEL-READY")

In [ ]:
# --------------------------------------------------
# Drop identifier
# --------------------------------------------------

df_ready = df_ready.drop(columns=['movie_id'])

In [ ]:
# --------------------------------------------------
#  Feature / target split
# --------------------------------------------------

X = df_ready.drop(columns=['worldwide_box_office'])
y = df_ready['worldwide_box_office']

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# --------------------------------------------------
# Convert boolean columns to 0/1
# --------------------------------------------------

bool_cols = df_ready.select_dtypes(include='bool').columns
df_ready[bool_cols] = df_ready[bool_cols].astype(int)

bool_cols = X.select_dtypes(include='bool').columns
X[bool_cols] = X[bool_cols].astype(int)

print(f"Converted {len(bool_cols)} boolean columns to int.")

print("Boolean columns converted in df_final.")

In [ ]:
# --------------------------------------------------
# Final checks
# --------------------------------------------------
#
print(X.dtypes.value_counts())
print("Total missing values:", X.isna().sum().sum())

# assert X.isna().sum().sum() == 0, "Missing values still exist!"
# assert len(X.select_dtypes(exclude=[np.number]).columns) == 0, \
#     "Non-numeric columns remain!"

print("FINAL CHECK PASSED — DATA IS MODEL-READY")

In [ ]:
# Now, I want to remove the movies that have 0 world wide box office and move them to another table. Code by Muayyad 
# Create a table with only movies that have a box office greater than 0
df_final_clean = df_ready[df_ready['worldwide_box_office'] > 0].copy()

# Move the rows with 0 box office to a separate "Zero Box Office" table
df_zero_box_office = df_ready[df_ready['worldwide_box_office'] == 0].copy()

# Verification
print(f"Movies with valid Box Office data: {df_final_clean.shape[0]}")
print(f"Movies moved to the Zero Table: {df_zero_box_office.shape[0]}")

# Check the minimum value in our new clean table to be sure
print(f"New minimum Box Office: ${df_final_clean['worldwide_box_office'].min()}")

In [ ]:
# --------------------------------------------------
# Save datasets
# --------------------------------------------------

import os

output_dir = "../final_data"
os.makedirs(output_dir, exist_ok=True)

# Training data
df_final_clean.to_csv(
    os.path.join(output_dir, "final_modeling_dataset.csv"),
    index=False
)

# Prediction data
df_zero_box_office.to_csv(
    os.path.join(output_dir, "zero_box_office_dataset.csv"),
    index=False
)

# Full master backup
df_master.to_csv(
    os.path.join(output_dir, "master_merged_dataset.csv"),
    index=False
)

print("FILES SAVED SUCCESSFULLY")
print("Final modeling dataset:", df_final_clean.shape)
print("Zero box office dataset:", df_zero_box_office.shape)
print("Master dataset:", df_master.shape)


### Target Variable Transformation (Worldwide Box Office)

Due to the highly skewed distribution of movie revenues, the prediction task is formulated as a classification problem by discretizing worldwide box office performance into quartile-based success categories.

The worldwide box office revenue exhibits a highly right-skewed distribution, with a small number of blockbuster movies generating disproportionately large revenues compared to the majority of films. Directly predicting continuous revenue values in such a setting can lead to unstable regression models that are overly influenced by extreme outliers.

To address this issue while preserving the economic importance of blockbuster movies, the prediction task is reformulated as a classification problem. Worldwide box office revenue is discretized into four performance categories. Movies in the top 10% of revenue are assigned to a dedicated “Blockbuster” class, ensuring that extreme but meaningful outcomes are not removed or diluted. The remaining 90% of movies are divided into three equally sized groups (Low, Medium, and High) using quantile-based bucketing to maintain class balance.

This approach reduces variance, improves model stability, and aligns predictions with business-relevant success tiers, while retaining the full spectrum of revenue outcomes in the dataset.


In [ ]:
#Stage 1: managing the predicted variable Y ---------------------------------Ahmed



# 0. blockbuster threshold
y_raw = df_final_clean['worldwide_box_office']

blockbuster_threshold = y_raw.quantile(0.90)
print(f"Blockbuster threshold (top 10%): {blockbuster_threshold:,.0f}")

# Create y_bucket as a nullable integer column (no dtype issues)
df_final_clean['y_bucket'] = pd.Series(pd.NA, index=df_final_clean.index, dtype="Int64")

# 1. Assign blockbuster class
df_final_clean.loc[
    df_final_clean['worldwide_box_office'] >= blockbuster_threshold,
    'y_bucket'
] = 3

# 2. Bucket the remaining 90% into 3 balanced classes
mask_non_blockbuster = df_final_clean['y_bucket'].isna()

q = pd.qcut(
    df_final_clean.loc[mask_non_blockbuster, 'worldwide_box_office'],
    q=3,
    duplicates='drop'   # safe even without zeros
)

# Convert qcut bins to integer codes 0,1,2 and assign
df_final_clean.loc[mask_non_blockbuster, 'y_bucket'] = q.cat.codes.astype("Int64")

# 3. Final type as regular int & print buckets(optional)
df_final_clean['y_bucket'] = df_final_clean['y_bucket'].astype(int)
df_final_clean['y_bucket'].value_counts().sort_index()

#4. Visualize the buckets

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_final_clean, x='y_bucket', y='worldwide_box_office')
plt.yscale('log')  # revenue is skewed, log scale makes it readable
plt.title('Worldwide Box Office by Bucket (log scale)')
plt.xlabel('Bucket (0=Low, 1=Medium, 2=High, 3=Blockbuster)')
plt.ylabel('Worldwide Box Office ($)')
plt.tight_layout()
plt.show()

#4. Create a classification column for each movie (row)
bucket_map = {
    0: 'Low',
    1: 'Medium',
    2: 'High',
    3: 'Blockbuster'
}

df_final_clean['y_class'] = df_final_clean['y_bucket'].map(bucket_map)
#Verification by prinitng few rows
df_final_clean[['worldwide_box_office', 'y_bucket', 'y_class']].head(10)

#5. Fix the predicted variable y for modeling ------ Ahmed
y = df_final_clean['y_bucket']



Since the prediction task is formulated as a classification problem, the target variable is represented as discrete classes rather than continuous values. Therefore, scaling or normalization is applied only to predictor variables. No transformations are applied to the target labels.


In [ ]:
# Stage 2: managing some features (Production Budget & Runtime)-----------------Ahmed


#Checks of scale and value for budget and runtime ----------------------------------------------------------

cols_to_check = ['production_budget', 'runtime'] # runtime is already imputed

missing_stats = (
    df_final_clean[cols_to_check]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

print("Missing value percentage (%):")
print(missing_stats)

# Scale of production budget
df_final_clean[cols_to_check].describe()

plt.figure(figsize=(6,4))
sns.histplot(df_final_clean['production_budget'], bins=50)
plt.xscale('log')
plt.title('Production Budget (log scale)')
plt.show()


from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# scaling budget & log transform (Production Budget)--------------------------------------------------------Ahmed

# Check for any missing budgets
num_zeros = (df_final_clean['production_budget'] == 0).sum()
print(f"Number of zero budgets: {num_zeros}")
print("Zeros:", (df_final_clean['production_budget'] == 0).sum())
print("NaNs:", df_final_clean['production_budget'].isna().sum())
print("Total rows:", len(df_final_clean))

# Create missing indicator
df_final_clean['budget_missing'] = (df_final_clean['production_budget'] == 0).astype(int)
import numpy as np

# Replace zeros with NaN
df_final_clean['production_budget'] = df_final_clean['production_budget'].replace(0, np.nan)

# Log-transform (budget is highly skewed)
df_final_clean['log_budget'] = np.log1p(df_final_clean['production_budget'])

# Median imputation on log scale
imp = SimpleImputer(strategy='median')
df_final_clean[['log_budget']] = imp.fit_transform(df_final_clean[['log_budget']])

# Scale the imnputed log transformed production budget
scaler = StandardScaler()
df_final_clean[['log_budget_scaled']] = scaler.fit_transform(df_final_clean[['log_budget']])
# (Optional safety) handle missing budgets
df_final_clean['production_budget'] = df_final_clean['production_budget'].fillna(
    df_final_clean['production_budget'].median()
)


# scaling imputed runtime (Runtime)--------------------------------------------------------------------------Ahmed
#Inspect the threshold
runtime_cap = df_final_clean['runtime'].quantile(0.99)
print(f"99th percentile runtime cap: {runtime_cap}")

#applying capping for outliers stabilization
df_final_clean['runtime_capped'] = df_final_clean['runtime'].clip(
    upper=runtime_cap
)

# Scale the capped imputed runtime
scaler_runtime = StandardScaler()
df_final_clean['runtime_scaled'] = scaler_runtime.fit_transform(
    df_final_clean[['runtime_capped']]
)

# (Optional safety) ensure no missing values remain:
df_final_clean['runtime'] = df_final_clean['runtime'].fillna(df_final_clean['runtime'].median())

#check
df_final_clean[['runtime', 'runtime_scaled', 'production_budget', 'log_budget', 'log_budget_scaled']].describe()

-------(Ahmed)--------- nummerical features

For Runtime:
shows a relatively stable distribution but contains a small number of extreme values, which are capped prior to scaling to prevent undue influence on distance-based models.
-->After capping extreme runtime values and applying standardization, the influence of unusually long films is substantially reduced. The scaled runtime variable exhibits a stable range, indicating successful outlier stabilization without removing any observations.

For production budget:
Production budget is highly right-skewed and is therefore log-transformed before standardization. A binary indicator is added to distinguish movies with missing or unreported budget values. (As 25% percentile of production_budget = 0 (1886))
-->Missing production budgets are handled using both a binary missingness indicator and log-scale median imputation, ensuring that the model retains information about missingness while still benefiting from a continuous budget feature.

In conclusion, distances are more stable and gradients are better prepared for.

Transformer input and output should be here along with their comments


For Dimensionality reduction:

SVD is applied exclusively to transformer-based text embeddings using Truncated SVD to extract latent semantic factors. Structured numeric and categorical features are retained in their engineered form to preserve interpretability. The final feature set combines reduced semantic features with structured predictors.

In [ ]:
#For Dimensionality reduction---------------------------------------Ahmed
#
# from sklearn.decomposition import TruncatedSVD
# #A, Reduce transformer embeddings (users & experts separately)-------Ahmed
#
#
# ############Compare 25 vs 50 vs 100 SVD components & Report explained variance############
# # svd_user = TruncatedSVD(n_components=50, random_state=42)
# # user_emb_reduced = svd_user.fit_transform(user_embeddings)
# # svd_expert = TruncatedSVD(n_components=50, random_state=42)
# # expert_emb_reduced = svd_expert.fit_transform(expert_embeddings)
#
# #B, Keep structured features unchanged
# X_structured = df_final_clean[
#     [
#         'runtime_scaled',
#         'log_budget_scaled',
#         'budget_missing',
#         'user_review_count',
#         'user_positive_ratio',
#         'user_sentiment_mean',
#         'expert_review_count',
#         'expert_positive_ratio',
#         'expert_sentiment_mean'
#     ]
# ].values
# #C, Combine into final feature matrix
# X_final = np.hstack([
#     X_structured
# ])


For Dataset spiliting into Train, Validation, and Test sets------------------Ahmed
-frist we need to check the balance pf the predicted variable y_bucket

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
#Visual check of class balance

y = df_final_clean['y_bucket']

class_counts = y.value_counts().sort_index()
class_percent = y.value_counts(normalize=True).sort_index() * 100

balance_df = pd.DataFrame({
    'count': class_counts,
    'percentage (%)': class_percent.round(2)
})


#Visual check of class balance
plt.figure(figsize=(6,4))
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title("Class distribution of y_bucket")
plt.xlabel("Sales bucket")
plt.ylabel("Number of movies")
plt.show()


The class distribution of the target variable was inspected after bucketing worldwide box office revenue. Buckets 0, 1, 2 are approximately balanced, while the blockbuster bucket (bucket 3) is smaller by design. This reflects the natural skew of movie revenues and does not cause a severe class imbalance. Therefore, no resampling techniques were applied.
--> Use stratified splitting not K-folds, as we have more than 4000 row values
--> Use stratified splitting not other methods that handels imbalanced data, such as class weighting, smote or others

dont run the next following until the SVD is done


In [ ]:
# Dataset splitting into Train, Validation, and Test sets------------------Ahmed

from sklearn.model_selection import train_test_split

X = X_final
y = df_final_clean['y_bucket'].values

# 1) 80% fitting, 20% testing
X_fit, X_test, y_fit, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# 2) From the 80% fitting: 80% training, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_fit, y_fit,
    test_size=0.20,          # 20% of the fitting set
    random_state=42,
    stratify=y_fit
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

# 3) datasets and objectives for modeling
train_set = (X_train, y_train)
val_set   = (X_val, y_val)
test_set  = (X_test, y_test)

# Verify class balance in each set
def balance(y_part):
    return pd.Series(y_part).value_counts(normalize=True).sort_index().round(3)

print("Train balance:\n", balance(y_train))
print("Val balance:\n", balance(y_val))
print("Test balance:\n", balance(y_test))
